# Clasificación de textos en esperanto: física, biología y química

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielaagonzalezz/cienciaNLP/blob/main/notebooks/clasificacion_esperanto.ipynb)

**Minería de Textos · 4º INF-A · CUNEF · 2026/2027**

**Equipo:** Daniela González ([@danielaagonzalezz](https://github.com/danielaagonzalezz)) · Olivia ([@Olivia029](https://github.com/Olivia029))

**Repositorio:** https://github.com/danielaagonzalezz/cienciaNLP

---

Este cuaderno es la **Fase 3** de la práctica. Usa lo que hicimos en las dos primeras, que está en el repositorio:

1. **Dataset** (`dataset/dataset.csv`): párrafos de la Wikipedia en esperanto etiquetados como `fiziko`, `biologio` o
   `kemio`. Lo genera el script `scripts/build_dataset.py`.
2. **Librería `ciencianlp`**: tokenizador, lematizador, POS-tagger y stop-words, basada en reglas y hecha desde cero.

Con ellos el cuaderno **vectoriza el corpus, entrena un clasificador, lo evalúa y clasifica páginas web reales**.

Hemos elegido física, biología y química porque son **temáticas cercanas**: comparten vocabulario científico
(*energio*, *molekulo*, *ĉelo*, *atomo*...), así que separarlas no es trivial.

**Independencia de temática:** el cuaderno no tiene las clases escritas en ningún sitio, las lee del dataset. Basta
con cambiar `URL_DATASET` por el dataset de otro equipo que cumpla el contrato de datos para reentrenarlo y evaluarlo.

## 0. Configuración

Todos los parámetros están en esta celda, para que el cuaderno se ejecute sin preguntar nada al usuario.

- `URL_DATASET`: dataset que se usa. Por defecto, el nuestro; se puede poner el de otro equipo.
- `URL_DATASET_RESPALDO`: copia del mismo fichero en otro servidor, por si el primero no responde.
- `SEMILLA`: se fija para que los resultados sean reproducibles.

In [ ]:
REPO = "danielaagonzalezz/cienciaNLP"
RAMA = "main"  # rama de la que se instala la librería y se lee el dataset

URL_DATASET = f"https://raw.githubusercontent.com/{REPO}/{RAMA}/dataset/dataset.csv"
URL_DATASET_RESPALDO = f"https://cdn.jsdelivr.net/gh/{REPO}@{RAMA}/dataset/dataset.csv"  # None si no hay copia

SEMILLA = 42

### Instalación

En Colab ya vienen `pandas`, `scikit-learn`, `matplotlib`, `requests` y `beautifulsoup4`. Solo falta nuestra
librería, que se instala con `pip` desde GitHub. Si ya está instalada, no se reinstala.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ciencianlp") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"git+https://github.com/{REPO}.git@{RAMA}"], check=True)

In [ ]:
import random
import re

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import sklearn

import ciencianlp

random.seed(SEMILLA)
np.random.seed(SEMILLA)

# Versiones usadas, para poder reproducir los resultados
for libreria in (ciencianlp, pd, np, sklearn, matplotlib, requests):
    print(f"{libreria.__name__:<12} {libreria.__version__}")

## 1. Dataset

### 1.1 Descarga

El dataset se descarga automáticamente del repositorio de GitHub. Para que el cuaderno no falle si GitHub no responde,
si la primera URL falla se prueba la copia del CDN jsDelivr, que sirve el mismo fichero del mismo repositorio.

In [ ]:
def descargar_dataset(*urls):
    """Lee el CSV de la primera URL que responda. Si ninguna responde, lanza un error claro."""
    for url in filter(None, urls):  # se salta las URL vacías (None)
        try:
            datos = pd.read_csv(url, sep=";", encoding="utf-8")
            print("Dataset leído de", url)
            return datos
        except Exception as error:  # sin red, 404, fichero corrupto...
            print("No se pudo leer", url, "->", error)
    raise RuntimeError("No se pudo descargar el dataset de ninguna fuente")


df = descargar_dataset(URL_DATASET, URL_DATASET_RESPALDO)
CLASES = sorted(df["class"].unique())  # las clases salen del dataset, no se escriben a mano
print("Clases:", CLASES)
df.head()

### 1.2 Comprobación del contrato de datos

El enunciado fija un contrato para el dataset. Lo comprobamos con `assert`, de forma que si algo no se cumple el
cuaderno se para y avisa:

- dos columnas con cabecera `text;class`,
- ningún párrafo vacío ni con el carácter `;`,
- al menos 3 clases,
- párrafos limpios: sin HTML, enlaces ni referencias del tipo `[1]`.

In [ ]:
assert list(df.columns) == ["text", "class"], "Las columnas deben ser text;class"
assert df["text"].notna().all() and (df["text"].str.strip() != "").all(), "Hay párrafos vacíos"
assert not df["text"].str.contains(";").any(), "Algún párrafo contiene ';'"
assert df["class"].nunique() >= 3, "Hacen falta al menos 3 clases"

restos = re.compile(r"<[a-z/][^>]*>|https?://|www\.|\[\d+\]")  # HTML, enlaces y referencias
assert not df["text"].str.contains(restos).any(), "Quedan restos de HTML, enlaces o referencias"

print(f"Contrato cumplido: {len(df)} párrafos, {df['class'].nunique()} clases")
df["class"].value_counts().rename("párrafos").to_frame()

## 2. Procesado con `ciencianlp`

Nuestra librería tiene una API parecida a la de spaCy: `nlp(texto)` devuelve un `Doc`, que se recorre token a token.
Cada `Token` trae su lema (`lemma_`), su etiqueta gramatical (`pos_`) y si es una stop-word (`is_stop`).

In [ ]:
from ciencianlp import nlp

ejemplo = nlp("La elektronoj de la atomo moviĝas ĉirkaŭ la kerno.")
pd.DataFrame([{"text": t.text, "lemma_": t.lemma_, "pos_": t.pos_, "is_stop": t.is_stop, "is_punct": t.is_punct}
              for t in ejemplo])

### 2.1 Preprocesado del corpus

De cada párrafo nos quedamos con los **lemas** de las palabras que aportan información sobre el tema:

- **Lematizar** junta las formas de una misma palabra: *ĉelo*, *ĉeloj*, *ĉelon* y *ĉelojn* cuentan como `ĉelo`. Así
  hay menos columnas en la tabla y cada una tiene más ejemplos.
- **Quitar stop-words** (*la*, *de*, *kaj*, *estas*...), porque aparecen igual en las tres clases.
- **Quitar signos y números** (`pos_ == "NUM"`): un año o una cifra no indica la temática.

Guardamos también las palabras sin lematizar (`palabras`), para comparar más adelante si la librería mejora el modelo.

In [ ]:
import time


def lemas_utiles(doc):
    """Lemas que aportan información temática: sin stop-words, signos ni números."""
    return [t.lemma_ for t in doc
            if not (t.is_stop or t.is_punct or t.pos_ == "NUM") and len(t.text) > 1]


inicio = time.perf_counter()
docs = [nlp(texto) for texto in df["text"]]
df["lemas"] = [lemas_utiles(doc) for doc in docs]
df["palabras"] = [[t.text.lower() for t in doc if not t.is_punct] for doc in docs]
print(f"{len(docs)} párrafos procesados en {time.perf_counter() - inicio:.1f} s")

formas = {t.text.lower() for doc in docs for t in doc if not t.is_punct}
lemas = {t.lemma_ for doc in docs for t in doc if not t.is_punct}
print(f"Vocabulario: {len(formas)} formas distintas -> {len(lemas)} lemas "
      f"({1 - len(lemas) / len(formas):.0%} menos gracias al lematizador)")

### 2.2 Etiquetas gramaticales del corpus

Porcentaje de tokens de cada categoría gramatical en cada clase. La fila `X` son las palabras que el etiquetador no
sabe clasificar (nombres propios sin terminación esperanta, siglas, fórmulas...); sirve para ver hasta dónde llegan
las reglas.

In [ ]:
etiquetas = pd.DataFrame([(clase, t.pos_) for doc, clase in zip(docs, df["class"]) for t in doc],
                         columns=["class", "pos"])
tabla_pos = pd.crosstab(etiquetas["pos"], etiquetas["class"], normalize="columns").mul(100).round(1)
tabla_pos.sort_values(tabla_pos.columns[0], ascending=False)

## 3. Exploración del dataset

### 3.1 Tamaño de las clases y longitud de los párrafos

Comprobamos que las clases están equilibradas (si una tuviera muchos más párrafos, el modelo tendería a predecirla) y
cómo de largos son los párrafos.

In [ ]:
df["n_palabras"] = df["text"].str.split().str.len()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
df["class"].value_counts().reindex(CLASES).plot.bar(ax=ax1, color="steelblue", rot=0)
ax1.set(title="Párrafos por clase", xlabel="", ylabel="párrafos")
for clase in CLASES:
    ax2.hist(df.loc[df["class"] == clase, "n_palabras"], bins=30, range=(0, 200), alpha=0.5, label=clase)
ax2.set(title="Longitud de los párrafos", xlabel="palabras", ylabel="párrafos")
ax2.legend()
plt.tight_layout()
plt.show()

df.groupby("class")["n_palabras"].describe()[["mean", "50%", "min", "max"]].round(0)

### 3.2 Lemas más frecuentes por clase

Los 15 lemas más frecuentes de cada clase (ya sin stop-words). Contamos en cuántos párrafos aparece cada lema, no
cuántas veces, para que un párrafo que repite mucho una palabra no pese de más.

In [ ]:
from collections import Counter

frecuentes = {clase: Counter(lema for lemas in df.loc[df["class"] == clase, "lemas"] for lema in set(lemas))
              for clase in CLASES}
pd.DataFrame({clase: [f"{lema} ({n})" for lema, n in frecuentes[clase].most_common(15)] for clase in CLASES})

In [ ]:
top = {clase: {lema for lema, _ in frecuentes[clase].most_common(50)} for clase in CLASES}
comunes = set.intersection(*top.values())
print(f"De los 50 lemas más frecuentes de cada clase, {len(comunes)} son comunes a todas:")
print(", ".join(sorted(comunes)))

Que muchos de los lemas más frecuentes sean comunes a todas las clases confirma que las temáticas son
**cercanas**: comparten el vocabulario general de la ciencia. Por eso conviene una representación como TF-IDF, que da
menos peso a las palabras que aparecen en todas partes.

## 4. Ruido en las etiquetas

Cada párrafo hereda la etiqueta del artículo del que sale, pero no todos los párrafos hablan del tema. Por ejemplo, un
artículo de física puede tener un párrafo sobre la historia de una universidad. Esos párrafos son **ruido**: llevan una
etiqueta que su contenido no justifica.

Para **medirlo** sin usar todavía ningún modelo, buscamos los **lemas distintivos** de cada clase: los que aparecen en
al menos `MIN_PARRAFOS` párrafos y, de esos párrafos, al menos un `UMBRAL` (70 %) son de una misma clase. Por ejemplo,
si `ĉelo` aparece en 40 párrafos y 36 son de `biologio`, es un lema distintivo de biología.

Después contamos, en cada párrafo, cuántos lemas distintivos tiene de su clase y cuántos de otra:

| Tipo | Condición | Qué significa |
|---|---|---|
| **temático** | más lemas de su clase que de cualquier otra | la etiqueta encaja con el contenido |
| **genérico** | ningún lema distintivo | no dice nada del tema (ruido) |
| **ambiguo** | empate entre su clase y otra | podría ser de las dos |
| **sospechoso** | más lemas de otra clase que de la suya | probablemente mal etiquetado (ruido) |

Para no hacer trampa, al estudiar un párrafo **se le quita de los recuentos**: un lema no puede ser distintivo solo
porque aparece en ese mismo párrafo.

In [ ]:
MIN_PARRAFOS = 5  # un lema tiene que salir en al menos 5 párrafos (sin contar el que se estudia)
UMBRAL = 0.7      # y el 70 % de ellos tienen que ser de la misma clase

# En cuántos párrafos aparece cada lema, en total y por clase
total = Counter()
por_clase = Counter()
for lemas_parrafo, clase in zip(df["lemas"], df["class"]):
    for lema in set(lemas_parrafo):
        total[lema] += 1
        por_clase[lema, clase] += 1


def clase_distintiva(lema, clase_propia):
    """Clase de la que es distintivo el lema sin contar el párrafo estudiado, o None."""
    n = total[lema] - 1
    if n < MIN_PARRAFOS:
        return None
    for clase in CLASES:
        apariciones = por_clase[lema, clase] - (clase == clase_propia)
        if apariciones / n >= UMBRAL:
            return clase
    return None


def tipo_de_parrafo(lemas_parrafo, clase_propia):
    votos = Counter(clase_distintiva(lema, clase_propia) for lema in set(lemas_parrafo))
    votos.pop(None, None)
    propios = votos.pop(clase_propia, 0)
    ajenos = max(votos.values(), default=0)
    if propios == ajenos == 0:
        return "genérico"
    if propios > ajenos:
        return "temático"
    return "ambiguo" if propios == ajenos else "sospechoso"


TIPOS = ["temático", "ambiguo", "genérico", "sospechoso"]
df["tipo"] = [tipo_de_parrafo(lemas_parrafo, clase) for lemas_parrafo, clase in zip(df["lemas"], df["class"])]

ruido = pd.crosstab(df["class"], df["tipo"], normalize="index").mul(100).reindex(columns=TIPOS).round(1)
ruido.loc["total"] = df["tipo"].value_counts(normalize=True).mul(100).reindex(TIPOS).round(1)
ruido

In [ ]:
ax = ruido.plot.barh(stacked=True, figsize=(10, 3.5), color=["seagreen", "gold", "darkgray", "indianred"])
ax.set(title="Tipo de párrafo según sus lemas distintivos", xlabel="% de párrafos", ylabel="")
ax.legend(bbox_to_anchor=(1, 1))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

ruidosos = df["tipo"].isin(["genérico", "sospechoso"]).mean()
print(f"Ruido estimado (genéricos + sospechosos): {ruidosos:.1%} de los párrafos")

### 4.1 Ejemplos

Algunos párrafos de cada tipo de ruido, para comprobar a mano que la medida tiene sentido.

In [ ]:
pd.set_option("display.max_colwidth", 220)
for tipo in ["genérico", "sospechoso"]:
    print(f"--- {tipo} ---")
    display(df.loc[df["tipo"] == tipo, ["class", "text"]].sample(4, random_state=SEMILLA))